# Install + Imports

In [ ]:
!pip install -q sentence-transformers scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import re

from google.colab import files
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity

#Upload Keywords

In [ ]:
uploaded = files.upload()
filename = list(uploaded.keys())[0]

df = pd.read_csv(filename)
assert "keyword" in df.columns, "CSV must contain a 'keyword' column"

df["keyword"] = df["keyword"].astype(str).str.lower().str.strip()
df.head()

Saving Mattress Insider Query Funnel Analysis - SAS_2026-01-09_21-09-05.csv to Mattress Insider Query Funnel Analysis - SAS_2026-01-09_21-09-05.csv


,keyword,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,alaskan king bed,NaN,NaN,NaN,NaN
1,mattress insider,NaN,NaN,NaN,NaN
2,family bed,NaN,NaN,NaN,NaN
3,heart shaped bed,NaN,NaN,NaN,NaN
4,alaskan king mattress,NaN,NaN,NaN,NaN


#Define Your Strategic Topics (Manual)

In [ ]:
MANUAL_TOPICS = {
    "RV Mattress": [
        "rv mattress",
        "custom rv mattress"
    ],
    "Alaskan King Mattress": [
        "Alaskan king bed",
        "alaskan king",
    ],
    "Custom Mattress": [
        "Custom Mattress",
        "Custom Mattress sizes",
    ],
    "Large Mattress": [
        "texas king bed",
        "wyoming king bed",
    ]
}

#Embed Topics & Keywords (Fast + Lightweight)

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

topic_embeddings = {}

for topic, phrases in MANUAL_TOPICS.items():
    embeds = model.encode(phrases, normalize_embeddings=True)
    topic_embeddings[topic] = embeds

In [ ]:
keyword_embeddings = model.encode(
    df["keyword"].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True
)

Batches:   0%|          | 0/782 [00:00<?, ?it/s]

#Attach Keywords to Topics (Similarity Gate)

In [ ]:
SIM_THRESHOLD = 0.65

In [ ]:
def assign_topic(keyword_embed):
    best_topic = None
    best_score = 0

    for topic, topic_embeds in topic_embeddings.items():
        sims = cosine_similarity(
            keyword_embed.reshape(1, -1),
            topic_embeds
        )[0]

        max_sim = sims.max()

        if max_sim > best_score:
            best_score = max_sim
            best_topic = topic

    if best_score >= SIM_THRESHOLD:
        return best_topic, round(best_score, 2)

    return None, round(best_score, 2)

#Apply Topic Assignment

In [ ]:
df[["assigned_topic", "topic_similarity"]] = [
    assign_topic(embed) for embed in keyword_embeddings
]


##drop null terms

df = df[df["assigned_topic"].notnull()].reset_index(drop=True)

#Buyer Journey Signals

In [ ]:
STAGES = {
    "Decision / Action": [
        r"\bbuy\b", r"\bpricing\b", r"\bdemo\b", r"\bconsultation\b",
        r"\bcontact\b", r"\bhire\b", r"\bsign up\b", r"\bget started\b"
    ],
    "Validation / Trust": [
        r"\breviews\b", r"\bcase study\b", r"\btestimonials\b",
        r"\blegit\b", r"\btrustworthy\b", r"\bworth it\b",
        r"\bis .* reliable\b", r"\bis .* accurate\b"
    ],
    "Narrowing / Evaluation": [
        r"\bbest\b", r"\bfeatures\b", r"\bcost\b",
        r"\bpricing\b", r"\bfor .*?\b", r"\bvs\b"
    ],
    "Exploration / Consideration": [
        r"\balternatives\b", r"\bcomparison\b",
        r"\bpros and cons\b", r"\btools\b", r"\bplatforms\b"
    ],
    "Trigger / Awareness": [
        r"\bwhat is\b", r"\bwhy\b", r"\bhow to\b",
        r"\bguide\b", r"\bideas\b", r"\bexamples\b"
    ]
}

STAGE_ORDER = list(STAGES.keys())

In [ ]:
df[["buyer_journey_stage", "confidence_score"]] = (
    df["keyword"]
    .apply(lambda k: pd.Series(classify_with_confidence(k)))
)

#Final Outputs

In [ ]:
df.head()

,keyword,topic_id,meta_topic,buyer_journey_stage,confidence_score,assigned_topic,topic_similarity
0,alaskan king bed,1,alaskan king bed,Trigger / Awareness,0.3,Alaskan King Mattress,1.0
1,alaskan king mattress,1,alaskan king bed,Trigger / Awareness,0.3,Alaskan King Mattress,0.91
2,alaskan king bed,1,alaskan king bed,Trigger / Awareness,0.3,Alaskan King Mattress,1.0
3,texas king mattress,1,alaskan king bed,Trigger / Awareness,0.3,Large Mattress,0.9
4,texas king bed,1,alaskan king bed,Trigger / Awareness,0.3,Large Mattress,1.0


#Export CSVs

In [ ]:
df.to_csv("keywords_by_topic_and_stage.csv", index=False)
files.download("keywords_by_topic_and_stage.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>